# RAG Ingestion Pipeline Verification

This notebook verifies the `RAGIngestion` class as required by the Stage 1 Prototyping rules.

In [1]:
import sys
import os
from google.cloud import bigquery, storage

# Ensure project root is in path
sys.path.append("../../..")

from pipelines.enterprise_knowledge_base.rag_ingestion import (
    RAGIngestion,
    IngestDocumentRequest
)

In [2]:
# Configuration
PROJECT_ID = "ag-core-dev-fdx7" # Update with your Project ID
os.environ["PROJECT_ID"] = PROJECT_ID

rag_pipeline = RAGIngestion()
print(f"RAG Pipeline initialized for project: {PROJECT_ID}")

2026-04-25 02:36:53.092 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:__init__:37 - Initialized RAGIngestion | CHUNK_SIZE: 1000 | OVERLAP: 100


RAG Pipeline initialized for project: ag-core-dev-fdx7


In [3]:
# Run the RAG Ingestion run() method
GCS_URI = "gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf"

try:
    req = IngestDocumentRequest(gcs_uri=GCS_URI)
    resp = rag_pipeline.run(req)
    print(f'Execution status: {resp.execution_status}')
    print(f'Chunks generated: {resp.chunk_count}')
    print(f'Final reported URI: {resp.processed_uri}')
except Exception as e:
    print(f'Pipeline execution error: {e}')

2026-04-25 02:36:53.098 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:run:54 - Starting end-to-end pipeline for: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf
2026-04-25 02:36:53.099 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:ingest_document:84 - Starting ingestion for document: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf
2026-04-25 02:36:56.045 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:_clear_existing_chunks:234 - Cleared 31 existing chunks for: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf
2026-04-25 02:36:56.045 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:_copy_to_staging:323 - Staging document: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf -> gs://ag-core-dev-fdx7-rag-staging/ingested/cb013450/SoW - Innovation.pdf
2026-04-25 02:37:00.930 | INFO     | pipelines.enterprise

Execution status: SUCCESS
Chunks generated: 15
Final reported URI: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf


In [4]:
resp.model_dump()

{'chunk_count': 15,
 'processed_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf',
 'execution_status': 'SUCCESS'}

In [5]:
# Verify BigQuery: Check if embeddings were generated
bq_client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT chunk_id, gcs_uri, ARRAY_LENGTH(embedding) as embedding_length 
FROM `{PROJECT_ID}.knowledge_base.documents_chunks` 
WHERE gcs_uri = '{GCS_URI}'
"""

results = bq_client.query(query).result()
for row in results:
     print(dict(row))

{'chunk_id': '40267ffb-0e60-43bf-aa9c-0bd71e6e0faf', 'gcs_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf', 'embedding_length': 1408}
{'chunk_id': '35e1956c-b2ba-4b61-bc1a-ab30212a9fc3', 'gcs_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf', 'embedding_length': 1408}
{'chunk_id': 'e728d2e8-e243-45e2-a636-fea25480dfb7', 'gcs_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf', 'embedding_length': 1408}
{'chunk_id': 'a397688f-7cc7-46ae-bf03-ae76a981b35f', 'gcs_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf', 'embedding_length': 1408}
{'chunk_id': '06560298-7e97-4153-b02c-b9a019ceff18', 'gcs_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf', 'embedding_length': 1408}
{'chunk_id': '5c9f1561-38b2-47f0-a7d4-b7d26a267acc', 'gcs_uri': 'gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf', 'embedding_length': 1408}
{'chunk_id': '01

## Staging Verification

In [6]:
storage_client = storage.Client()
STAGING_BUCKET = f"{PROJECT_ID}-rag-staging"
FILENAME = GCS_URI.split('/')[-1]

print(f"Verifying isolation in staging bucket: {STAGING_BUCKET}...")
blob = storage_client.bucket(STAGING_BUCKET).get_blob(f"processed/{FILENAME}")

if blob:
    print(f"SUCCESS: Processed copy found in internal staging: {blob.name}")
    print("INFO: Domain bucket remains untouched (read-only reference).")
else:
    print(f"FAILURE: Could not find processed file in {STAGING_BUCKET}/processed/")

Verifying isolation in staging bucket: ag-core-dev-fdx7-rag-staging...
FAILURE: Could not find processed file in ag-core-dev-fdx7-rag-staging/processed/


In [7]:
request = IngestDocumentRequest(
    gcs_uri=GCS_URI,
    original_uri=GCS_URI
)
# 5. Ejecutar la ingesta (Chunking + BQ Load)
print(f"Iniciando ingesta para: {GCS_URI}...")
response = rag_pipeline.ingest_document(request)
# 6. Mostrar resultados
print("\n--- Resultados de Ingesta ---")
print(f"Status: {response.execution_status}")
print(f"Chunks creados y almacenados: {response.chunk_count}")
print(f"URI de registro: {response.processed_uri}")
if response.chunk_count > 0:
    print("\n✅ Los chunks ya están en BigQuery listos para ser vectorizados.")
else:
    print("\n⚠️ No se generaron chunks. Revisa los logs para más detalles.")

2026-04-25 02:37:07.575 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:ingest_document:84 - Starting ingestion for document: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf


Iniciando ingesta para: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf...


2026-04-25 02:37:09.972 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:_clear_existing_chunks:234 - Cleared 15 existing chunks for: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf
2026-04-25 02:37:09.973 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:_copy_to_staging:323 - Staging document: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf -> gs://ag-core-dev-fdx7-rag-staging/ingested/05833a79/SoW - Innovation.pdf
2026-04-25 02:37:14.551 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:ingest_document:107 - Successfully staged 15 chunks to BigQuery.
2026-04-25 02:37:14.551 | DEBUG    | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:_move_blob_to_processed:357 - Moving staging file ingested/05833a79/SoW - Innovation.pdf to processed/05833a79/SoW - Innovation.pdf



--- Resultados de Ingesta ---
Status: SUCCESS
Chunks creados y almacenados: 15
URI de registro: gs://kb-it/confidential/orchestration-test/tester/SoW - Innovation.pdf

✅ Los chunks ya están en BigQuery listos para ser vectorizados.
